# High-Resolution Satellite Image Compression — Colab training
Run cells top to bottom. Set Runtime > Change runtime type > GPU first.

In [ ]:
!git clone <YOUR_REPO_URL_HERE> satcomp
%cd satcomp
!pip install -q -r requirements.txt

## 1. Smoke test (no dataset needed, ~10 seconds)

In [ ]:
!python scripts/smoke_test.py

## 2. Fetch EuroSAT (primary dataset — small, fast, fits free-tier GPU hours)

In [ ]:
import torchvision
torchvision.datasets.EuroSAT(root='.', download=True)
DATA_ROOT = './eurosat/2750'

## 3. Train (start small: few epochs, small model, to validate the pipeline before a long run)

In [ ]:
!python -m src.train \
  --data_root {DATA_ROOT} \
  --max_patch_size 64 --min_patch_size 16 --split_threshold 0.1 \
  --embed_dim 128 --depth 4 --num_heads 4 --latent_dim 64 \
  --epochs 10 --lam 0.01 --accum_steps 16 \
  --out_dir checkpoints_lam0.01

## 4. Rate-distortion sweep (repeat step 3 with different --lam, --out_dir)
Suggested lambdas: 0.001, 0.005, 0.01, 0.05, 0.1

## 5. Evaluate + fixed-grid ablation (the key novelty comparison)

In [ ]:
!python -m src.evaluate \
  --checkpoint checkpoints_lam0.01/model_epoch9.pt \
  --data_root {DATA_ROOT} --max_patch_size 64 --min_patch_size 16 \
  --fixed_grid_ablation --limit 50 --out_json eval_lam0.01.json

## 6. Inspect results / plot rate-distortion curve

In [ ]:
import json
import matplotlib.pyplot as plt

results = []  # fill in as (bpp, psnr) pairs after running step 5 for each lambda
with open('eval_lam0.01.json') as f:
    d = json.load(f)
print(json.dumps(d['summary'], indent=2))